<a href="https://colab.research.google.com/github/Tejesh95/RAG-ingestion-pipeline/blob/main/RAG_ingestion_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
PDF / PPTX / DOCX / TXT
          ↓
   Smart Parser
          ↓
   Text Extraction
          ↓
   Cleaning + Metadata
          ↓
   Chunking
          ↓
   Gemini Embeddings
          ↓
   Qdrant Cloud
          ↓
   Retrieve Top-K
          ↓
   FlashRank Reranking
          ↓
   Top Relevant Chunks

In [ ]:
!pip install -q pypdf python-docx python-pptx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.6 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
from pathlib import Path

from pypdf import PdfReader
from docx import Document
from pptx import Presentation

In [ ]:
def create_document(text, source, file_type, page=None, slide=None):
    return {
        "text": text,
        "source": source,
        "file_type": file_type,
        "page": page,
        "slide": slide
    }

In [ ]:
def parse_pdf(file_path):
    reader = PdfReader(file_path)

    documents = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        if text.strip():
            documents.append(
                create_document(
                    text=text,
                    source=os.path.basename(file_path),
                    file_type="pdf",
                    page=page_number
                )
            )

    return documents

In [ ]:
pdf_docs = parse_pdf("/content/Tejesh_resume21.pdf")

print("Pages extracted:", len(pdf_docs))
print(pdf_docs[0]["text"][:1000])

Pages extracted: 1
Indukuru Tejesh
n210594@rguktn.ac.in|+91-8309584905|github.com/Tejesh95|linkedin.com/in/tejesh95
Professional Summary
Computer Science undergraduate with hands-on Python and Java programming experience, applied research in AI/ML, and a track
record of shipping full-stack systems spanning APIs, databases, and cloud-integrated services. Strong foundation in object-oriented
design, problem solving, and data structures, with proven ability to learn new stacks quickly and collaborate effectively in team
settings.
Education
Rajiv Gandhi University of Knowledge T echnologies (RGUKT)Expected 2027
Bachelor of Technology in Computer Science and Engineering
• Relevant Coursework: Object-Oriented Programming, Data Structures & Algorithms, Database Management Systems, Machine
Learning, Computer Vision, Data Analytics
• CS50 Python Certified (Harvard/edX)
Technical Skills
Programming Languages: Python, Java, JavaScript, HTML
F rameworks & T echnologies: FastAPI, Streamlit, REST AP

In [ ]:
def parse_docx(file_path):
    doc = Document(file_path)

    documents = []

    # Extract paragraphs
    paragraphs = []

    for paragraph in doc.paragraphs:
        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    # Extract tables
    tables = []

    for table in doc.tables:
        for row in table.rows:
            row_text = " | ".join(
                cell.text.strip()
                for cell in row.cells
            )

            if row_text.strip():
                tables.append(row_text)

    all_text = "\n".join(paragraphs + tables)

    if all_text.strip():
        documents.append(
            create_document(
                text=all_text,
                source=os.path.basename(file_path),
                file_type="docx"
            )
        )

    return documents

In [ ]:
docx_docs = parse_docx("/content/Medsam_Report.docx")

print(docx_docs[0]["text"][:1000])

RAJIV GANDHI UNIVERSITY OF KNOWLEDGE TECHNOLOGIES
Nuzvid, Andhra Pradesh
Department of Computer Science and Engineering
A MINI-PROJECT REPORT
on
Reshaping the Data, Not the Model:
Submitted by
Rayi Mohan Vinay Raj (N210974)
Ch Enosh (N210966)
S Naseer Ali (N210948)
Moyyi Durga Prasad (N210944)
Under the Guidance of
Mr. A. Udaya Kumar
Assistant Professor & Head of the Department of Computer Science and Engineering
April 2026
RAJIV GANDHI UNIVERSITY OF KNOWLEDGE TECHNOLOGIES, NUZVID
(ACT 18 OF 2008, ANDHRA PRADESH)
DEPARTMENT OF COMPUTER SCIENCE ENGINEERING
Nuzvid, Eluru, Andhra Pradesh – 521202.
___________________________________________________________________________
CERTIFICATE OF COMPLETION
This is to certify that the work entitled, "Reshaping the Data, Not the Model: Latent Refinement for Foundation Medical Segmentation Without Parametric Updates" is the Bonafide work of Rayi Mohan Vinay Raj (N210974) , Ch Enosh (N210966), S Naseer Ali (N210948), Moyyi Durga Prasad (N210944) carri

In [ ]:
def parse_pptx(file_path):
    presentation = Presentation(file_path)

    documents = []

    for slide_number, slide in enumerate(
        presentation.slides,
        start=1
    ):
        slide_text = []

        for shape in slide.shapes:
            if hasattr(shape, "text"):
                text = shape.text.strip()

                if text:
                    slide_text.append(text)

        combined_text = "\n".join(slide_text)

        if combined_text.strip():
            documents.append(
                create_document(
                    text=combined_text,
                    source=os.path.basename(file_path),
                    file_type="pptx",
                    slide=slide_number
                )
            )

    return documents

In [ ]:
pptx_docs = parse_pptx("/content/FUNCTION ORIENTED SOFTWARE DESIGN.pptx")

print("Slides extracted:", len(pptx_docs))
print(pptx_docs[0]["text"][:1000])

Slides extracted: 26
FUNCTION ORIENTED SOFTWARE DESIGN
SOFTWARE ENGINEERING COURSE


In [ ]:
def parse_txt(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    return [
        create_document(
            text=text,
            source=os.path.basename(file_path),
            file_type="txt"
        )
    ]

In [ ]:
def smart_parser(file_path):
    extension = Path(file_path).suffix.lower()

    if extension == ".pdf":
        return parse_pdf(file_path)

    elif extension == ".docx":
        return parse_docx(file_path)

    elif extension == ".pptx":
        return parse_pptx(file_path)

    elif extension == ".txt":
        return parse_txt(file_path)

    else:
        raise ValueError(
            f"Unsupported file type: {extension}"
        )

In [ ]:
file_paths = [
    "/content/Tejesh_resume21.pdf",
    "/content/Medsam_Report.docx",
    "/content/FUNCTION ORIENTED SOFTWARE DESIGN.pptx"
]

all_documents = []

for file_path in file_paths:
    docs = smart_parser(file_path)
    all_documents.extend(docs)

print("Total extracted document units:", len(all_documents))

Total extracted document units: 28


In [ ]:
def clean_text(text):
    # Replace multiple spaces with one
    text = re.sub(r"[ \t]+", " ", text)

    # Replace excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [ ]:
for document in all_documents:
    document["text"] = clean_text(document["text"])

In [ ]:
def chunk_text(text, chunk_size=800, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [ ]:
def create_chunks(documents):
    chunked_documents = []

    for document in documents:
        chunks = chunk_text(document["text"])

        for chunk_index, chunk in enumerate(chunks):
            chunked_documents.append({
                "text": chunk,
                "source": document["source"],
                "file_type": document["file_type"],
                "page": document["page"],
                "slide": document["slide"],
                "chunk_index": chunk_index
            })

    return chunked_documents

In [ ]:
chunked_documents = create_chunks(all_documents)

print("Total chunks:", len(chunked_documents))

Total chunks: 77


In [ ]:
print(json.dumps(chunked_documents[0], indent=2))

{
  "text": "Indukuru Tejesh\nn210594@rguktn.ac.in|+91-8309584905|github.com/Tejesh95|linkedin.com/in/tejesh95\nProfessional Summary\nComputer Science undergraduate with hands-on Python and Java programming experience, applied research in AI/ML, and a track\nrecord of shipping full-stack systems spanning APIs, databases, and cloud-integrated services. Strong foundation in object-oriented\ndesign, problem solving, and data structures, with proven ability to learn new stacks quickly and collaborate effectively in team\nsettings.\nEducation\nRajiv Gandhi University of Knowledge T echnologies (RGUKT)Expected 2027\nBachelor of Technology in Computer Science and Engineering\n\u2022 Relevant Coursework: Object-Oriented Programming, Data Structures & Algorithms, Database Management Systems, Machine\nLearning, Computer Vision,",
  "source": "Tejesh_resume21.pdf",
  "file_type": "pdf",
  "page": 1,
  "slide": null,
  "chunk_index": 0
}


In [ ]:
print("Total extracted document units:", len(all_documents))
print("Total chunks:", len(chunked_documents))

Total extracted document units: 28
Total chunks: 77


In [ ]:
Document chunk
      ↓
Gemini Embedding
      ↓
[0.012, -0.084, 0.231, ...]
      ↓
Qdrant Cloud

In [ ]:
!pip install -q google-genai qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 2.8 MB/s eta 0:00:00


In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("Tejesh_Key")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized")

Gemini client initialized


In [ ]:
response = client.models.embed_content(
    model="gemini-embedding-001",
    contents="Machine learning is a subset of artificial intelligence."
)

embedding = response.embeddings[0].values

print("Embedding dimensions:", len(embedding))
print("First 10 values:", embedding[:10])

Embedding dimensions: 3072
First 10 values: [-0.026853403, -0.0049841655, -0.016345479, -0.046370346, -0.02222475, -0.008440223, 0.00544005, 0.0032457092, 0.0038371943, 0.0025398768]


In [ ]:
EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 768

In [ ]:
def embed_text(text):
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config={
            "output_dimensionality": EMBEDDING_DIMENSION
        }
    )

    return response.embeddings[0].values

In [ ]:
test_vector = embed_text("What is deep learning?")

print("Vector length:", len(test_vector))
print("First 5 values:", test_vector[:5])

Vector length: 768
First 5 values: [-0.005311402, -0.0008777652, 0.008303766, -0.058969364, -0.01614367]


In [ ]:
from qdrant_client import QdrantClient
from google.colab import userdata

QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

print("Qdrant client initialized")

Qdrant client initialized


In [ ]:
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "rag_documents"

In [ ]:
if not qdrant.collection_exists(COLLECTION_NAME):
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=EMBEDDING_DIMENSION,
            distance=Distance.COSINE
        )
    )

print("Collection ready:", COLLECTION_NAME)

Collection ready: rag_documents


In [ ]:
from tqdm.auto import tqdm

for document in tqdm(chunked_documents):
    document["embedding"] = embed_text(document["text"])

print("Embeddings generated:", len(chunked_documents))
print("Dimension:", len(chunked_documents[0]["embedding"]))

  0%|          | 0/77 [00:00<?, ?it/s]

Embeddings generated: 77
Dimension: 768


WHAT HAPPENED

77 chunks

    ↓
77 Gemini API calls

    ↓
77 vectors[link text](https://)

In [ ]:
from qdrant_client.models import PointStruct #upload into qdrant

In [ ]:
points = []

for idx, document in enumerate(chunked_documents):
    points.append(
        PointStruct(
            id=idx,
            vector=document["embedding"],
            payload={
                "text": document["text"],
                "source": document["source"],
                "file_type": document["file_type"],
                "page": document["page"],
                "slide": document["slide"],
                "chunk_index": document["chunk_index"]
            }
        )
    )

In [ ]:
qdrant.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print("Uploaded points:", len(points))

Uploaded points: 77


In [43]:
collection_info = qdrant.get_collection(COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Vector count:", collection_info.points_count)

Collection: rag_documents
Vector count: 77


In [44]:
print("Vector count:", collection_info.points_count)
print("Dimension:", len(chunked_documents[0]["embedding"]))

Vector count: 77
Dimension: 768


Phase 3


In [ ]:
User Query
    │
    ▼
Gemini Query Embedding
    │
    ▼
Qdrant Semantic Search
    │
    ▼
Top 10–20 Candidate Chunks
    │
    ▼
FlashRank Reranker
    │
    ▼
Top 3 Most Relevant Chunks

In [45]:
!pip install -q flashrank

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 36.2 MB/s eta 0:00:00


In [47]:
from flashrank import Ranker, RerankRequest

In [48]:
ranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",
    cache_dir="/tmp"
)

ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:00<00:00, 23.8MiB/s]


In [49]:
def embed_query(query):
    return embed_text(query)

In [50]:
query = "What is machine learning?"
query_vector = embed_query(query)

print("Query vector dimension:", len(query_vector))

Query vector dimension: 768


In [51]:
def semantic_search(query, top_k=10):
    query_vector = embed_query(query)

    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True
    ).points

    return results

In [52]:
query = "What is machine learning?"

results = semantic_search(query, top_k=10)

print("Results returned:", len(results))

Results returned: 10


In [53]:
for i, result in enumerate(results, start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Score:", result.score)
    print("Source:", result.payload["source"])
    print("Page:", result.payload["page"])
    print("Text:", result.payload["text"][:500])

Rank: 1
Score: 0.5571003
Source: Medsam_Report.docx
Page: None
Text: .... 19
4.3 Results ................................................................................................................ 21
4.4 Error Analysis .................................................................................................... 22
5 CONCLUSION AND FUTURE WORK ............................................................. 23
5.1 Conclusion ......................................................................................................... 23
5.2 Future Work .......
Rank: 2
Score: 0.55135113
Source: Medsam_Report.docx
Page: None
Text: ............... 25
CHAPTER 1
Introduction
The rapid advancement of deep learning has enabled the development of large-scale foundation models capable of performing generalised medical image segmentation. Models such as MedSAM (Medical Segment Anything Model) are trained on vast repositories of labelled medical images and demonstrate remarkable zero-shot perf

In [54]:
def prepare_rerank_passages(results):
    passages = []

    for result in results:
        passages.append({
            "id": str(result.id),
            "text": result.payload["text"],
            "source": result.payload["source"],
            "page": result.payload["page"],
            "slide": result.payload["slide"],
            "chunk_index": result.payload["chunk_index"]
        })

    return passages

In [55]:
def rerank_results(query, results, top_k=3):
    passages = prepare_rerank_passages(results)

    rerank_request = RerankRequest(
        query=query,
        passages=passages
    )

    reranked_results = ranker.rerank(rerank_request)

    return reranked_results[:top_k]

In [56]:
query = "What is machine learning?"

results = semantic_search(query, top_k=10)

reranked_results = rerank_results(
    query,
    results,
    top_k=3
)

print("Reranked results:", len(reranked_results))

Reranked results: 3


In [57]:
for i, result in enumerate(reranked_results, start=1):
    print("=" * 80)
    print("Final Rank:", i)
    print("Rerank Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Text:", result["text"][:700])

Final Rank: 1
Rerank Score: 0.29112798
Source: Medsam_Report.docx
Page: None
Text: ning Research, 13(25), 723–773.
Lafferty, J., McCallum, A., & Pereira, F. C. N. (2001). Conditional random fields: Probabilistic models for segmenting and labelling sequence data. In Proceedings of the 18th International Conference on Machine Learning (ICML), 282–289.
Grandvalet, Y., & Bengio, Y. (2004). Semi-supervised learning by entropy minimisation. Advances in Neural Information Processing Systems (NeurIPS), 17.
Kirillov, A., Mintun, E., Ravi, N., Mao, H., Rolland, C., Gustafson, L., ... & Girshick, R. (2023). Segment anything. In Proceedings of the IEEE/CVF International Conference on Computer Vision (ICCV), 4015–4026.
Mr. A. Udaya Kumar,
Assistant Professor, 
Head of Department, 
Depa
Final Rank: 2
Rerank Score: 0.0005631373
Source: Medsam_Report.docx
Page: None
Text: ............................... 8
1.2 Expected Outcomes ...........................................................................

In [58]:
def retrieve_and_rerank(query, retrieval_k=10, final_k=3):
    # 1. Retrieve candidates from Qdrant
    results = semantic_search(query, top_k=retrieval_k)

    # 2. Rerank candidates using FlashRank
    reranked_results = rerank_results(
        query,
        results,
        top_k=final_k
    )

    return reranked_results

In [59]:
query = "What is the difference between supervised and unsupervised learning?"

final_results = retrieve_and_rerank(
    query,
    retrieval_k=10,
    final_k=3
)

for i, result in enumerate(final_results, start=1):
    print("=" * 80)
    print("Final Rank:", i)
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Text:", result["text"][:700])

Final Rank: 1
Score: 0.006436577
Source: Medsam_Report.docx
Page: None
Text: endencies. The proposed DAL-CRF extends this concept by applying CRF principles directly within the latent embedding space, preserving structural topology within the adapted representation.
Grandvalet and Bengio, 2004 proposed entropy minimisation as a semi-supervised learning principle, demonstrating that minimising output distribution entropy over unlabelled data implicitly encourages low-density decision boundaries. This principle is adapted here for test-time adaptation, with a key modification to target only foreground entropy.
2.1 Comparison of Existing Methods
Table 1 provides a structured comparison of existing test-time adaptation approaches against the proposed Latent Refinement me
Final Rank: 2
Score: 0.0024224736
Source: Medsam_Report.docx
Page: None
Text: ning Research, 13(25), 723–773.
Lafferty, J., McCallum, A., & Pereira, F. C. N. (2001). Conditional random fields: Probabilistic models for segm

phase 4

In [ ]:
FlashRank Results
       ↓
Context Builder
       ↓
Gemini LLM
       ↓
Grounded Answer

In [61]:
def build_context(results):
    context_parts = []

    for i, result in enumerate(results, start=1):
        source = result.get("source", "Unknown")
        page = result.get("page")
        slide = result.get("slide")

        location = ""

        if page is not None:
            location = f"Page {page}"
        elif slide is not None:
            location = f"Slide {slide}"

        context_parts.append(
            f"[Context {i} | Source: {source} | {location}]\n"
            f"{result['text']}"
        )

    return "\n\n".join(context_parts)

In [62]:
context = build_context(final_results)

print(context[:2000])

[Context 1 | Source: Medsam_Report.docx | ]
endencies. The proposed DAL-CRF extends this concept by applying CRF principles directly within the latent embedding space, preserving structural topology within the adapted representation.
Grandvalet and Bengio, 2004 proposed entropy minimisation as a semi-supervised learning principle, demonstrating that minimising output distribution entropy over unlabelled data implicitly encourages low-density decision boundaries. This principle is adapted here for test-time adaptation, with a key modification to target only foreground entropy.
2.1 Comparison of Existing Methods
Table 1 provides a structured comparison of existing test-time adaptation approaches against the proposed Latent Refinement method.
Figure 3: Computational wall of current test-time adaptation methods. Latent Updates (Proposed

[Context 2 | Source: Medsam_Report.docx | ]
ning Research, 13(25), 723–773.
Lafferty, J., McCallum, A., & Pereira, F. C. N. (2001). Conditional random fie

In [63]:
def create_rag_prompt(query, context):
    return f"""
You are a helpful question-answering assistant.

Answer the user's question using only the provided context.

Rules:
1. Use the context as the primary source of truth.
2. Do not invent facts that are not present in the context.
3. If the answer cannot be found in the context, say:
   "I couldn't find that information in the provided documents."
4. Give a clear and concise answer.
5. Mention the relevant source when possible.

Context:
--------------------
{context}
--------------------

Question:
{query}

Answer:
"""

In [64]:
client = genai.Client(api_key=GEMINI_API_KEY)

In [68]:
def generate_answer(query, context):
    prompt = create_rag_prompt(query, context)

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

In [66]:
def rag_query(query, retrieval_k=10, final_k=3):
    # 1. Retrieve candidates from Qdrant
    results = semantic_search(
        query,
        top_k=retrieval_k
    )

    # 2. Rerank candidates using FlashRank
    reranked_results = rerank_results(
        query,
        results,
        top_k=final_k
    )

    # 3. Build context
    context = build_context(reranked_results)

    # 4. Generate grounded answer
    answer = generate_answer(
        query,
        context
    )

    return {
        "query": query,
        "answer": answer,
        "sources": reranked_results
    }

In [69]:
result = rag_query(
    "What is machine learning?"
)

print(result["answer"])

I couldn't find that information in the provided documents.


In [70]:
print("ANSWER")
print("=" * 80)
print(result["answer"])

print("\nSOURCES")
print("=" * 80)

for i, source in enumerate(result["sources"], start=1):
    print(f"{i}. {source['source']}")

    if source["page"] is not None:
        print(f"   Page: {source['page']}")

    if source["slide"] is not None:
        print(f"   Slide: {source['slide']}")

    print(f"   Rerank score: {source['score']}")

ANSWER
I couldn't find that information in the provided documents.

SOURCES
1. Medsam_Report.docx
   Rerank score: 0.2911279797554016
2. Medsam_Report.docx
   Rerank score: 0.0005631372914649546
3. Medsam_Report.docx
   Rerank score: 0.000543303438462317
